# SG-VO — LoRA Online Adaptation Experiment
### ICRA Research Notebook

**Before running:** `Runtime → Change runtime type → T4 GPU`

This notebook runs four conditions and compares them in a table:
| # | Condition | Params updated |
|---|---|---|
| A | Offline VO (no adaptation) | 0 |
| B | Full online adaptation (paper) | 39M |
| C | LoRA-attention (rank 8) | 21K (0.08%) |
| D | LoRA-attention + trigger (0.8) | 21K, ~50% frames |

In [ ]:
# ── Cell 1: GPU Check ────────────────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Clone Repo and Install Dependencies ──────────────────────────────
import os
if not os.path.exists('/content/SG-VO'):
    !git clone https://github.com/AreebaAzizRajput/SG-VO.git /content/SG-VO
else:
    %cd /content/SG-VO
    !git pull
%cd /content/SG-VO
!pip install -q einops path imageio scikit-image tqdm gdown
print('✅ Done')

In [ ]:
# ── Cell 3: Download Pretrained Checkpoints ──────────────────────────────────
import os, glob, shutil
os.makedirs('checkpoints', exist_ok=True)

if not os.path.exists('checkpoints/exp_pose112_model_best.pth.tar'):
    print('Downloading pretrained models from Google Drive...')
    !gdown --folder 'https://drive.google.com/drive/folders/1twZsg2pxMLwcUUnFszBn1ryaEtuoEfs3' \
        -O checkpoints/ --quiet
    # flatten any subfolder structure
    for f in glob.glob('checkpoints/**/*.pth.tar', recursive=True):
        dest = os.path.join('checkpoints', os.path.basename(f))
        if f != dest: shutil.move(f, dest)

!ls -lh checkpoints/*.pth.tar

In [ ]:
# ── Cell 4: Download Camera Intrinsics ───────────────────────────────────────
import os
os.makedirs('data/kitti_odom/sequences', exist_ok=True)

if not os.path.exists('data/kitti_odom/sequences/kitti_odom256_intrinsics/cam_09.txt'):
    print('Downloading camera intrinsics...')
    !gdown --folder 'https://drive.google.com/drive/folders/1n81QDHaG3lIxnxybl9I6knPHxPngTsD8' \
        -O data/kitti_odom/sequences/ --quiet

!ls data/kitti_odom/sequences/kitti_odom256_intrinsics/

In [ ]:
# ── Cell 5: Download KITTI Odometry Sequences 09 and 10 ─────────────────────
# Full zip is ~65 GB. We extract only sequences 09 and 10 (~12 GB).
import os, shutil

DATASET_DIR = 'data/kitti_odom/sequences'
total, used, free = shutil.disk_usage('/')
print(f'Disk: {free//1e9:.0f} GB free')

zip_path = '/tmp/data_odometry_color.zip'

if not os.path.exists(f'{DATASET_DIR}/09/image_2'):
    if not os.path.exists(zip_path):
        print('Downloading KITTI (~65 GB, ~10-15 min on Colab)...')
        !wget -q --show-progress -c \
            'https://s3.eu-central-1.amazonaws.com/avg-kitti/data_odometry_color.zip' \
            -O {zip_path}
    print('Extracting sequences 09 and 10...')
    !unzip -q {zip_path} 'dataset/sequences/09/*' 'dataset/sequences/10/*' -d /tmp/kitti_raw/
    !mv /tmp/kitti_raw/dataset/sequences/09 {DATASET_DIR}/
    !mv /tmp/kitti_raw/dataset/sequences/10 {DATASET_DIR}/
    !rm -f {zip_path}
    print('✅ Done')
else:
    print('✅ Already present')

import glob
for seq in ['09','10']:
    n = len(glob.glob(f'{DATASET_DIR}/{seq}/image_2/*.png'))
    print(f'  Seq {seq}: {n} images')

In [ ]:
# ── Cell 6: Install cam.txt Intrinsics for Each Sequence ─────────────────────
import os, shutil

INTR_DIR = 'data/kitti_odom/sequences/kitti_odom256_intrinsics'
SEQ_DIR  = 'data/kitti_odom/sequences'

for seq in ['09', '10']:
    src  = f'{INTR_DIR}/cam_{seq}.txt'
    dest = f'{SEQ_DIR}/{seq}/image_2/cam.txt'
    if os.path.exists(src):
        shutil.copy(src, dest)
        print(f'✅ {dest}')
    else:
        print(f'⚠️  Missing {src}')

In [ ]:
# ── Cell 7: Sanity Check — LoRA Injection ────────────────────────────────────
import sys; sys.path.insert(0, '/content/SG-VO')
from lora import inject_lora_pose_net, freeze_all, lora_parameters, count_lora_params
from models import PoseResNet
import torch

pose_net = PoseResNet(num_layers=50, pretrained=False)
n_total  = sum(p.numel() for p in pose_net.parameters())
freeze_all(pose_net)
inject_lora_pose_net(pose_net, rank=8, alpha=1.0, targets='attention')
n_lora = count_lora_params(pose_net)
print(f'LoRA params: {n_lora:,} / {n_total:,}  ({100*n_lora/n_total:.3f}%)')

# quick forward pass
with torch.no_grad():
    out = pose_net(torch.randn(1,36,256,832), torch.randn(1,36,256,832))
print(f'Forward pass OK — output shape: {out.shape}')
print('✅ LoRA ready')

## Condition A — Offline VO (baseline, no adaptation)

In [ ]:
# ── Cell 8: Run Offline VO ───────────────────────────────────────────────────
import os
os.makedirs('vo_results_offline', exist_ok=True)

POSE_NET = 'checkpoints/exp_pose112_model_best.pth.tar'
DATASET  = 'data/kitti_odom/sequences/'

for seq in ['09', '10']:
    print(f'\n=== Offline VO — Sequence {seq} ===')
    !python test_vo.py \
        --img-height 256 --img-width 832 \
        --sequence {seq} \
        --pretrained-posenet {POSE_NET} \
        --dataset-dir {DATASET} \
        --output-dir vo_results_offline/

print('\n=== Offline results ===')
!python kitti_eval/eval_odom.py --result=vo_results_offline/ --align=7dof

## Condition B — Full Online Adaptation (original paper)

In [ ]:
# ── Cell 9: Run Full Online VO ───────────────────────────────────────────────
import os
os.makedirs('vo_results_full', exist_ok=True)

POSE_NET = 'checkpoints/exp_pose112_model_best.pth.tar'
DISP_NET = 'checkpoints/dispnet112_model_best.pth.tar'
DATASET  = 'data/kitti_odom/sequences/'

for seq in ['09', '10']:
    print(f'\n=== Full Online VO — Sequence {seq} ===')
    !python test_vo_online.py \
        -p 1 -s 0.1 -c 0.5 \
        --img-height 256 --img-width 832 \
        --sequence {seq} \
        --pretrained-posenet {POSE_NET} \
        --pretrained-disp    {DISP_NET} \
        --dataset-dir {DATASET} \
        --output-dir  vo_results_full/ \
        --epochs 2 --lr 1e-4 \
        --sequence-length 3 \
        --resnet-layers 50 \
        --select-best 1 \
        --with-mask 1 --with-auto-mask 1 \
        --padding-mode border

print('\n=== Full online results ===')
!python kitti_eval/eval_odom.py --result=vo_results_full/ --align=7dof

## Condition C — LoRA-Attention Adaptation (new contribution)

In [ ]:
# ── Cell 10: Run LoRA Online VO ──────────────────────────────────────────────
import os
os.makedirs('vo_results_lora', exist_ok=True)

POSE_NET = 'checkpoints/exp_pose112_model_best.pth.tar'
DISP_NET = 'checkpoints/dispnet112_model_best.pth.tar'
DATASET  = 'data/kitti_odom/sequences/'

for seq in ['09', '10']:
    print(f'\n=== LoRA Online VO — Sequence {seq} ===')
    !python test_vo_online.py \
        -p 1 -s 0.1 -c 0.5 \
        --img-height 256 --img-width 832 \
        --sequence {seq} \
        --pretrained-posenet {POSE_NET} \
        --pretrained-disp    {DISP_NET} \
        --dataset-dir {DATASET} \
        --output-dir  vo_results_lora/ \
        --epochs 2 --lr 1e-4 \
        --sequence-length 3 \
        --resnet-layers 50 \
        --select-best 1 \
        --with-mask 1 --with-auto-mask 1 \
        --padding-mode border \
        --lora-rank 8 \
        --lora-targets attention

print('\n=== LoRA results ===')
!python kitti_eval/eval_odom.py --result=vo_results_lora/ --align=7dof

## Condition D — LoRA + Trigger (efficiency variant)

In [ ]:
# ── Cell 11: Run LoRA + Trigger ──────────────────────────────────────────────
import os
os.makedirs('vo_results_lora_trigger', exist_ok=True)

POSE_NET = 'checkpoints/exp_pose112_model_best.pth.tar'
DISP_NET = 'checkpoints/dispnet112_model_best.pth.tar'
DATASET  = 'data/kitti_odom/sequences/'

for seq in ['09', '10']:
    print(f'\n=== LoRA + Trigger — Sequence {seq} ===')
    !python test_vo_online.py \
        -p 1 -s 0.1 -c 0.5 \
        --img-height 256 --img-width 832 \
        --sequence {seq} \
        --pretrained-posenet {POSE_NET} \
        --pretrained-disp    {DISP_NET} \
        --dataset-dir {DATASET} \
        --output-dir  vo_results_lora_trigger/ \
        --epochs 2 --lr 1e-4 \
        --sequence-length 3 \
        --resnet-layers 50 \
        --select-best 1 \
        --with-mask 1 --with-auto-mask 1 \
        --padding-mode border \
        --lora-rank 8 \
        --lora-targets attention \
        --adapt-threshold 0.8

print('\n=== LoRA + Trigger results ===')
!python kitti_eval/eval_odom.py --result=vo_results_lora_trigger/ --align=7dof

## Results Comparison Table

In [ ]:
# ── Cell 12: Comparison Table ────────────────────────────────────────────────
import subprocess, re

PAPER = {'offline': {'09': (7.08,2.48), '10': (8.72,3.11)},
         'online':  {'09': (5.21,1.93), '10': (6.74,2.57)}}

def parse_eval(result_dir):
    """Run eval_odom.py and parse t_err, r_err for seq 09 and 10."""
    out = subprocess.run(['python','kitti_eval/eval_odom.py',
                          f'--result={result_dir}','--align=7dof'],
                         capture_output=True, text=True).stdout
    results = {}
    for seq in ['09','10']:
        t = re.search(rf'{seq}.*?([0-9]+\.[0-9]+).*?([0-9]+\.[0-9]+)', out)
        if t:
            results[seq] = (float(t.group(1)), float(t.group(2)))
        else:
            results[seq] = (None, None)
    return results

experiments = [
    ('A  Offline VO',          'vo_results_offline'),
    ('B  Full online (paper)',  'vo_results_full'),
    ('C  LoRA-attn rank 8',     'vo_results_lora'),
    ('D  LoRA + trigger 0.8',   'vo_results_lora_trigger'),
]

print(f'{'Condition':<28} | {'Seq 09 t%':>9} {'r°/100m':>8} | {'Seq 10 t%':>9} {'r°/100m':>8}')
print('-'*70)
# Paper reference rows
print(f"{'  Paper offline':28} | {PAPER['offline']['09'][0]:9.2f} {PAPER['offline']['09'][1]:8.2f} | {PAPER['offline']['10'][0]:9.2f} {PAPER['offline']['10'][1]:8.2f}")
print(f"{'  Paper online':28} | {PAPER['online']['09'][0]:9.2f} {PAPER['online']['09'][1]:8.2f} | {PAPER['online']['10'][0]:9.2f} {PAPER['online']['10'][1]:8.2f}")
print('-'*70)
for name, d in experiments:
    import os
    if not os.path.exists(d): continue
    r = parse_eval(d)
    t09, r09 = r.get('09',(None,None))
    t10, r10 = r.get('10',(None,None))
    t09s = f'{t09:9.2f}' if t09 else '      N/A'
    r09s = f'{r09:8.2f}' if r09 else '     N/A'
    t10s = f'{t10:9.2f}' if t10 else '      N/A'
    r10s = f'{r10:8.2f}' if r10 else '     N/A'
    print(f'{name:<28} | {t09s} {r09s} | {t10s} {r10s}')

## Trajectory Visualization

In [ ]:
# ── Cell 13: Trajectory Plots ────────────────────────────────────────────────
exec(open('scripts/colab_eval_viz.py').read())

In [ ]:
# ── Cell 14: Download All Results ────────────────────────────────────────────
import zipfile, os
from google.colab import files

with zipfile.ZipFile('/content/sgvo_lora_results.zip', 'w') as z:
    for d in ['vo_results_offline','vo_results_full',
              'vo_results_lora','vo_results_lora_trigger']:
        if not os.path.exists(d): continue
        for f in os.listdir(d):
            z.write(os.path.join(d, f))

files.download('/content/sgvo_lora_results.zip')